# Neural Identifier Training with Particle Filters - Differential Robot System

In [11]:
# Neural Identifier Training with Particle Filters - Differential Robot System

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time

# ============================================================
# 1) True nonlinear system (Differential Robot)
# ============================================================
def plant_dynamics(x_state, u):
    """
    Continuous dynamics for Differential Drive Robot.
    x_state = [x, y, theta] (position and orientation)
    u = [v, omega] (linear and angular velocities)
    
    Equations:
    dx/dt = v * cos(theta)
    dy/dt = v * sin(theta)
    dtheta/dt = omega
    
    Parameters:
    - x, y: position in 2D plane (m)
    - theta: orientation angle (rad)
    - v: linear velocity (m/s)
    - omega: angular velocity (rad/s)
    """
    x, y, theta = x_state
    v, omega = u
    
    dx_dt = v * np.cos(theta)
    dy_dt = v * np.sin(theta)
    dtheta_dt = omega
    
    return np.array([dx_dt, dy_dt, dtheta_dt])

def plant(x_k, u_k, dt=0.01, process_noise_type='gaussian', process_noise_std=1e-2):
    """
    One Euler step of the discrete plant with ADVANCED process noise.
    """
    # Simple Euler integration
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot

    # Initialize noise container
    noise = np.zeros_like(x_kp1)

    # --- NOISE GENERATION ---
    if process_noise_type == 'laplacian':
        # Peakier than Gaussian, heavier tails
        noise = np.random.laplace(0, process_noise_std, size=x_kp1.shape)

    elif process_noise_type == 'uniform':
        # Bounded noise
        a = np.sqrt(3) * process_noise_std
        noise = np.random.uniform(-a, a, size=x_kp1.shape)

    elif process_noise_type == 'student_t':
        # HEAVY TAILS: Good for testing robustness to outliers
        # df=3 is very heavy-tailed. As df -> infinity, it becomes Gaussian.
        df = 3 
        # Standardize so the spread is roughly comparable to process_noise_std
        noise = np.random.standard_t(df, size=x_kp1.shape) * process_noise_std

    elif process_noise_type == 'cauchy':
        # EXTREME OUTLIERS: The "EKF Killer". Undefined variance.
        noise = np.random.standard_cauchy(size=x_kp1.shape) * process_noise_std

    elif process_noise_type == 'bimodal':
        # MULTI-MODAL: The "Ambiguity" test.
        # Randomly choose between two peaks: +2*std and -2*std
        # This creates a distribution with two humps, confusing linear filters.
        means = np.random.choice([1, -1], size=x_kp1.shape)
        gauss_part = np.random.normal(0, process_noise_std * 0.5, size=x_kp1.shape)
        # Shift the noise to be centered around +2*std or -2*std
        noise = (means * 2 * process_noise_std) + gauss_part

    elif process_noise_type == 'impulsive':
        # SHOT NOISE: Mostly Gaussian, but with occasional massive spikes
        # Base gaussian noise
        noise = np.random.normal(0, process_noise_std, size=x_kp1.shape)
        # 5% chance of a massive spike (10x standard deviation)
        prob_spike = 0.05
        mask = np.random.choice([0, 1], size=x_kp1.shape, p=[1-prob_spike, prob_spike])
        spikes = np.random.normal(0, 10 * process_noise_std, size=x_kp1.shape)
        noise += mask * spikes

    else:  # gaussian (Default)
        noise = np.random.normal(0, process_noise_std, size=x_kp1.shape)

    return x_kp1 + noise

# ============================================================
# 2) RHONN structure (Updated for 3 States) - SIMPLIFIED
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    z = np.clip(z, -10, 10)
    return 1.0 / (1.0 + np.exp(-beta * z))

# ============================================================
# SIMPLIFIED RHONN CONFIGURATIONS FOR DIFFERENTIAL ROBOT (3 states)
# ============================================================

def rhonn_config_1(x_est, scale=0.5):
    """
    Config 1: Minimal - Linear + Sigmoid
    Best for simple differential robot dynamics
    """
    x, y, theta = x_est
    s1 = sigmoidal(x * scale)
    s2 = sigmoidal(y * scale)
    s3 = sigmoidal(theta * scale)
    
    return np.array([
        s1,              # sigmoid(x)
        s2,              # sigmoid(y)
        s3,              # sigmoid(theta)
        x,               # linear x
        y,               # linear y
        1.0              # bias
    ])

def rhonn_config_2(x_est, scale=0.5):
    """
    Config 2: Standard - Interactions + Quadratic
    Good balance for nonlinear robot dynamics
    """
    x, y, theta = x_est
    s1 = sigmoidal(x * scale)
    s2 = sigmoidal(y * scale)
    s3 = sigmoidal(theta * scale)
    
    return np.array([
        s1,              # sigmoid(x)
        s2,              # sigmoid(y)
        s3,              # sigmoid(theta)
        s1 * s2,         # interaction x-y
        s1 * s3,         # interaction x-theta
        s2 * s3,         # interaction y-theta
        1.0              # bias
    ])

def rhonn_config_3(x_est, scale=0.5):
    """
    Config 3: Extended - More nonlinear terms
    For complex robot behavior
    """
    x, y, theta = x_est
    s1 = sigmoidal(x * scale)
    s2 = sigmoidal(y * scale)
    s3 = sigmoidal(theta * scale)
    
    return np.array([
        s1,              # sigmoid(x)
        s2,              # sigmoid(y)
        s3,              # sigmoid(theta)
        s1 * s2,         # interaction x-y
        s1 * s3,         # interaction x-theta
        s2 * s3,         # interaction y-theta
        s1**2,           # sigmoid^2(x)
        s2**2,           # sigmoid^2(y)
        s3**2,           # sigmoid^2(theta)
        1.0              # bias
    ])


# Dictionary of available configurations
RHONN_CONFIGS = {
    1: {'func': rhonn_config_1, 'name': 'Minimal (Linear + Sigmoid)', 'n_features': 6},
    2: {'func': rhonn_config_2, 'name': 'Standard (Interactions)', 'n_features': 7},
    3: {'func': rhonn_config_3, 'name': 'Extended (More Nonlinear)', 'n_features': 10},
}

# Global variable to store selected configuration
SELECTED_RHONN_CONFIG = None

def set_rhonn_config(config_id=1):
    """
    Set the active RHONN configuration
    
    Parameters:
    -----------
    config_id : int or str
        Configuration ID (1, 2, 3, or 'custom')
    """
    global SELECTED_RHONN_CONFIG
    if config_id not in RHONN_CONFIGS:
        raise ValueError(f"Invalid config_id: {config_id}. Choose from {list(RHONN_CONFIGS.keys())}")
    
    SELECTED_RHONN_CONFIG = config_id
    config = RHONN_CONFIGS[config_id]
    print(f"\n{'='*60}")
    print(f"🔧 RHONN Configuration Set:")
    print(f"   ID: {config_id}")
    print(f"   Name: {config['name']}")
    print(f"   Number of features: {config['n_features']}")
    print(f"{'='*60}\n")
    
    return config['n_features']

def construct_z_vector(x_est, scale=0.5):
    """
    Construct feature vector using the selected RHONN configuration
    
    Parameters:
    -----------
    x_est : array
        State estimate [x, y, theta]
    scale : float
        Scaling factor for sigmoid inputs
        
    Returns:
    --------
    array : Feature vector
    """
    if SELECTED_RHONN_CONFIG is None:
        raise ValueError("RHONN configuration not set! Call set_rhonn_config() first.")
    
    config_func = RHONN_CONFIGS[SELECTED_RHONN_CONFIG]['func']
    return config_func(x_est, scale)

def print_rhonn_configs():
    """Print all available RHONN configurations"""
    print("\n" + "="*70)
    print("📋 AVAILABLE RHONN CONFIGURATIONS:")
    print("="*70)
    for config_id, config in RHONN_CONFIGS.items():
        print(f"\nConfig {config_id}: {config['name']}")
        print(f"   Features: {config['n_features']}")
        # Show sample output
        sample_state = np.array([0.5, 1.0, 0.3])  # x=0.5, y=1.0, theta=0.3
        sample_z = config['func'](sample_state, scale=0.5)
        print(f"   Sample z (for [x=0.5, y=1.0, theta=0.3]): {sample_z}")
    print("="*70 + "\n")

def RHONN_predict(x_state_for_z, w_neuron):
    """
    Predicts next-state component with a single RHONN neuron.
    """
    z_i = construct_z_vector(x_state_for_z)
    if len(z_i) != len(w_neuron):
        raise ValueError(f"Dimension mismatch: z({len(z_i)}) vs w({len(w_neuron)})")
    return np.dot(w_neuron, z_i)

# ============================================================
# 3) EKF trainer over weights
# ============================================================
class EKF_RHONN_Trainer:
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-1, R_init=1e-2, P_init=1.0, eta=1.0):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta
        self.weights, self.P, self.Q, self.R = [], [], [], []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.05
            self.weights.append(w_i)
            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def update(self, chi_kp1, chi_k, x_hat_previous):
        # Series-Parallel: Use measured true state (chi_k) to build feature vector
        x_state_for_z = np.copy(chi_k) 

        z_i = construct_z_vector(x_state_for_z)          
        H_i = z_i.reshape(-1, 1)                          

        for i in range(self.num_neurons):
            P_pred = self.P[i] + self.Q[i] + np.eye(self.num_weights_per_neuron) * 1e-8
            
            M_i = self.R[i][0] + (H_i.T @ P_pred @ H_i)[0, 0]
            if M_i < 1e-10: M_i = 1e-10

            x_hat_pred_i = self.weights[i] @ z_i
            e_i = chi_kp1[i] - x_hat_pred_i
            # e_i = np.clip(e_i, -20.0, 20.0)

            K_i = (P_pred @ H_i).flatten() / M_i

            eta = self.eta * (1.0 / (1.0 + np.abs(e_i) * 0.01))
            self.weights[i] += eta * K_i * e_i

            self.P[i] = (np.eye(self.num_weights_per_neuron) - np.outer(K_i, H_i.ravel())) @ P_pred
            
            # PSD enforcement
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            if np.min(np.linalg.eigvals(self.P[i])) <= 0:
                self.P[i] += np.eye(self.num_weights_per_neuron) * 1e-5

In [12]:
# 4) Particle Filter trainer with OPTIONAL OPTIMIZATION
# ============================================================
# ✨ NEW FEATURE: Gradient-based optimization for top-k particles
#    - Refines best particles using gradient descent
#    - Improves convergence with 2000+ particles
#    - Optional: enable_optimization=True
# ============================================================

class PF_RHONN_Trainer:
    """
    Particle Filter trainer for RHONN weights (FULLY VECTORIZED).
    
    Parameters:
    -----------
    num_neurons : int
        Number of neurons (states to estimate)
    num_weights_per_neuron : int
        Number of weights per neuron
    n_particles : int
        Number of particles (recommended: 2000 for optimization)
    initial_weights : list of arrays, optional
        Initial weight values for each neuron
    Q_std : float or array-like
        Process noise standard deviation
    R_std : float or array-like
        Measurement noise standard deviation
    ess_threshold : float, optional
        Effective Sample Size threshold for resampling
    enable_optimization : bool, optional
        Enable gradient-based particle optimization (default: False)
    opt_learning_rate : float, optional
        Learning rate for gradient descent (default: 0.01)
    opt_top_k : int, optional
        Number of top particles to optimize (default: 10)
    """
    
    def __init__(self, num_neurons, num_weights_per_neuron, n_particles=500,
                 initial_weights=None, Q_std=0.5, R_std=0.5, ess_threshold=None,
                 enable_optimization=False, opt_learning_rate=0.01, opt_top_k=10):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.n_particles = n_particles
        self.enable_optimization = enable_optimization
        self.opt_learning_rate = opt_learning_rate
        self.opt_top_k = min(opt_top_k, n_particles)
        
        # Convert Q_std to per-neuron array
        if np.isscalar(Q_std):
            self.Q_std = np.ones(num_neurons) * Q_std
        else:
            self.Q_std = np.array(Q_std)
            if len(self.Q_std) != num_neurons:
                raise ValueError(f"Q_std debe tener {num_neurons} elementos")
            
        # Convert R_std to variance per-neuron array
        if np.isscalar(R_std):
            self.R_var = np.ones(num_neurons) * (R_std**2)
            self.R_std = np.ones(num_neurons) * R_std
        else:
            R_std_array = np.array(R_std)
            if len(R_std_array) != num_neurons:
                raise ValueError(f"R_std debe tener {num_neurons} elementos")
            self.R_std = R_std_array
            self.R_var = R_std_array**2
            
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 2.0

        # Initialize particles: (num_neurons, n_particles, num_weights_per_neuron)
        if initial_weights is not None:
            base_weights = np.array([np.copy(initial_weights[i]) if i < len(initial_weights) 
                                    else np.random.randn(num_weights_per_neuron) * 0.05 
                                    for i in range(num_neurons)])
            
            self.particles = (base_weights[:, np.newaxis, :] + 
                            np.random.randn(num_neurons, n_particles, num_weights_per_neuron) * 0.05)
        else:
            self.particles = np.random.randn(num_neurons, n_particles, num_weights_per_neuron) * 0.05

        # Weights: (num_neurons, n_particles)
        self.weights_pf = np.ones((num_neurons, n_particles)) / n_particles

    def _ess_vectorized(self, w):
        """Calculate ESS for all neurons (vectorized)."""
        sum_w = np.sum(w, axis=1, keepdims=True)
        sum_w = np.where(sum_w == 0, 1.0, sum_w)
        w_norm = w / sum_w
        return 1.0 / np.sum(w_norm**2, axis=1)

    def _ess(self, w):
        """Calculate ESS for single neuron (kept for compatibility)."""
        sum_w = np.sum(w)
        if sum_w == 0:
            return 0
        w_norm = w / sum_w
        return 1.0 / np.sum(w_norm**2)

    def _resample_systematic(self, neuron_index):
        """Systematic resampling (optimized with searchsorted)."""
        w = self.weights_pf[neuron_index]
        p = self.particles[neuron_index]
        
        # Normalize
        w = w / np.sum(w)
        N = len(w)
        
        # Systematic resampling with searchsorted
        u0 = np.random.uniform(0.0, 1.0 / N)
        positions = u0 + np.arange(N) / N
        cdf = np.cumsum(w)
        indexes = np.searchsorted(cdf, positions)
        indexes = np.clip(indexes, 0, N - 1)
        
        # Resample
        self.particles[neuron_index] = p[indexes]
        self.weights_pf[neuron_index] = np.ones(N) / N
    
    def _optimize_particles(self, neuron_idx, chi_kp1_i, z):
        """
        Gradient-based optimization for top-k particles of a neuron.
        
        Parameters:
        -----------
        neuron_idx : int
            Index of neuron to optimize
        chi_kp1_i : float
            Target measurement for this neuron
        z : array
            Feature vector
        """
        # Select top-k particles based on weights
        top_indices = np.argpartition(self.weights_pf[neuron_idx], -self.opt_top_k)[-self.opt_top_k:]
        
        # Get particles to optimize
        particles_opt = self.particles[neuron_idx, top_indices]  # (k, num_weights)
        
        # Gradient descent step
        # Prediction: y = w^T z
        predictions = particles_opt @ z  # (k,)
        errors = predictions - chi_kp1_i  # (k,)
        
        # Gradient: dL/dw = (y - target) * z
        gradients = np.outer(errors, z)  # (k, num_weights)
        
        # Update particles with gradient descent
        particles_opt -= self.opt_learning_rate * gradients
        
        # Put optimized particles back
        self.particles[neuron_idx, top_indices] = particles_opt

    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        Update particle filter (FULLY VECTORIZED).
        
        Parameters:
        -----------
        chi_kp1 : array (num_neurons,)
        chi_k : array (num_neurons,)
        x_hat_previous : array (not used in series-parallel)
        """
        # Build feature vector
        z = construct_z_vector(chi_k)  # (num_weights,)

        # ===== PREDICT (VECTORIZED) =====
        noise = np.random.randn(self.num_neurons, self.n_particles, self.num_weights_per_neuron)
        self.particles += noise * self.Q_std[:, np.newaxis, np.newaxis]

        # ===== UPDATE (VECTORIZED) =====
        # Predictions: (num_neurons, n_particles)
        x_pred_particles = np.einsum('ijk,k->ij', self.particles, z)
        
        # Innovation: (num_neurons, n_particles)
        innov = chi_kp1[:, np.newaxis] - x_pred_particles
        
        # Likelihood: (num_neurons, n_particles)
        ll = -0.5 * (innov**2) / self.R_var[:, np.newaxis]
        ll -= np.max(ll, axis=1, keepdims=True)
        like = np.exp(ll) + 1e-400
        
        # Update weights
        self.weights_pf *= like
        self.weights_pf /= np.sum(self.weights_pf, axis=1, keepdims=True)

        # ===== RESAMPLE =====
        ess_all = self._ess_vectorized(self.weights_pf)
        needs_resampling = ess_all < self.ess_threshold
        
        for i in np.where(needs_resampling)[0]:
            self._resample_systematic(i)

        # ===== OPTIMIZATION (OPTIONAL) =====
        if self.enable_optimization:
            for i in range(self.num_neurons):
                self._optimize_particles(i, chi_kp1[i], z)

    def get_estimate(self):
        """Get weight estimates (vectorized)."""
        # Weighted average: sum(w_ij * p_ij) for each neuron
        # weights_pf: (num_neurons, n_particles)
        # particles: (num_neurons, n_particles, num_weights)
        estimates = np.sum(
            self.weights_pf[:, :, np.newaxis] * self.particles,
            axis=1
        )  # (num_neurons, num_weights)
        
        return [estimates[i] for i in range(self.num_neurons)]

    def get_statistics(self):
        """Get statistics (vectorized)."""
        ess = self._ess_vectorized(self.weights_pf)
        stats = {
            'ess': ess.tolist(),
            'ess_ratio': (ess / self.n_particles).tolist(),
            'max_weight': np.max(self.weights_pf, axis=1).tolist(),
            'min_weight': np.min(self.weights_pf, axis=1).tolist()
        }
        return stats
    
    def get_info(self):
        """Get PF configuration information."""
        info = f"\n🔬 Particle Filter Configuration:\n"
        info += f"   Particles: {self.n_particles}\n"
        info += f"   Optimization: {'✅ ENABLED' if self.enable_optimization else '❌ Disabled'}\n"
        if self.enable_optimization:
            info += f"      - Learning Rate: {self.opt_learning_rate}\n"
            info += f"      - Top-K Particles: {self.opt_top_k}\n"
        return info

In [13]:

# ============================================================
# 5) Simulation
# ============================================================
if __name__ == "__main__":
    # ============================================================
    # 🔧 SET RHONN CONFIGURATION HERE
    # ============================================================
    # Choose configuration: 1, 2, or 3
    # - Config 1: Minimal (Linear + Sigmoid) - 6 features
    # - Config 2: Standard (Interactions) - 7 features  ⭐ RECOMMENDED
    # - Config 3: Extended (More Nonlinear) - 10 features
    
    RHONN_CONFIG_ID = 2  # 👈 CHANGE THIS TO SELECT CONFIGURATION
    num_weights_per_neuron = set_rhonn_config(RHONN_CONFIG_ID)
    
    # ============================================================
    
    # --- Reproducibility seed ---
    # SEED = np.random.randint(0, 10000)
    SEED = 7517
    np.random.seed(SEED)
    print(f"🎲 Semilla aleatoria (seed): {SEED}")
    print("   (Para reproducibilidad de resultados)\n")
    
    # --- Simulation settings ---
    n_steps = 1000
    dt = 0.02
    t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

    process_noise_type = 'gaussian' # 'gaussian', 'laplacian', 'uniform', 'student_t', 'cauchy', 'bimodal', 'impulsive'
    process_noise_std = 0.02

    measurement_noise_std = 0.05

    # Filter parameters for 3 states: x, y, theta
    Q_std_per_neuron = [1.0, 1.0, 1.0]  # x, y, theta
    R_std_per_neuron = [0.005, 0.005, 0.005]  # x, y, theta

    # --- True system init ---
    x_true = np.zeros((n_steps, 3))
    x_true[0] = [0.0, 0.0, 0.0]  # Start at origin, facing East
    
    # --- Control inputs (trajectory generation) ---
    # Create a circular trajectory by varying linear and angular velocities
    u_history = np.zeros((n_steps, 2))  # [v, omega]
    
    for k in range(n_steps):
        t = k * dt
        # Circular path with varying speed
        v = 0.5 + 0.3 * np.sin(0.5 * t)  # Linear velocity: 0.2-0.8 m/s
        omega = 0.3 * np.cos(0.3 * t)    # Angular velocity: ±0.3 rad/s
        u_history[k] = [v, omega]
    
    # --- RHONN config ---
    num_neurons = 3  # x, y, theta

    num_particles = 1500

    # --- Initial weights ---
    common_initial_weights = [np.random.uniform(-1.0, 1.0, num_weights_per_neuron) for _ in range(num_neurons)]

    # --- Instantiate Trainers ---
    ekf_trainer = EKF_RHONN_Trainer(num_neurons, num_weights_per_neuron, initial_weights=common_initial_weights, 
                                    eta=1.0, Q_init=1e-3, R_init=1e-7, P_init=2.0)
    pf_trainer = PF_RHONN_Trainer(num_neurons, num_weights_per_neuron, n_particles=num_particles, 
                                  initial_weights=common_initial_weights, 
                                  Q_std=Q_std_per_neuron, R_std=R_std_per_neuron, ess_threshold=0.5*num_particles,
                                  enable_optimization=True, opt_learning_rate=0.001*num_particles, opt_top_k=int(0.025*num_particles))


    # Initialize Particle Filter cloud
    for i in range(num_neurons):
        pf_trainer.particles[i] = np.tile(common_initial_weights[i], (pf_trainer.n_particles, 1))
        pf_trainer.particles[i] += np.random.normal(size=(pf_trainer.n_particles, num_weights_per_neuron)) * 0.1

    # Print PF configuration
    print(pf_trainer.get_info())

    # Storage
    x_hat_ekf = np.zeros((n_steps, 3))
    x_hat_ekf[0] = x_true[0]
    x_hat_pf = np.zeros((n_steps, 3))
    x_hat_pf[0] = x_true[0]

    # Timing variables
    ekf_time_total = 0.0
    pf_time_total = 0.0

    print("Starting Differential Robot simulation...")
    for k in range(n_steps - 1):
        # 1) Evolve true system with control input
        u_k = u_history[k]
        x_true[k+1] = plant(x_true[k], u_k, dt, process_noise_type, process_noise_std) + \
                      np.random.normal(0, measurement_noise_std, size=3)

        # Build series-parallel features from measured state at time k
        chi_k = x_true[k]
        chi_kp1 = x_true[k+1]

        # 2) EKF - with timing
        t_start_ekf = time.perf_counter()
        ekf_trainer.update(chi_kp1, chi_k, x_hat_ekf[k])
        # Prediction for next step (Series-Parallel)
        z_ekf = construct_z_vector(chi_k)
        for i in range(3): 
            x_hat_ekf[k+1, i] = np.dot(ekf_trainer.weights[i], z_ekf)
        t_end_ekf = time.perf_counter()
        ekf_time_total += (t_end_ekf - t_start_ekf)

        # 3) PF - with timing
        t_start_pf = time.perf_counter()
        pf_trainer.update(chi_kp1, chi_k, x_hat_pf[k])
        est_w_pf = pf_trainer.get_estimate()
        z_pf = construct_z_vector(chi_k)
        for i in range(3): 
            x_hat_pf[k+1, i] = np.dot(est_w_pf[i], z_pf)
        t_end_pf = time.perf_counter()
        pf_time_total += (t_end_pf - t_start_pf)

    print("Simulación Completa.")
    
    # Display timing results
    print("\n" + "="*70)
    print("⏱️  TIEMPOS DE ENTRENAMIENTO:")
    print("="*70)
    print(f"EKF Total:     {ekf_time_total:.4f} segundos  ({ekf_time_total*1000:.2f} ms)")
    print(f"EKF Por Paso:  {ekf_time_total/(n_steps-1)*1000:.4f} ms/paso")
    print(f"\nPF Total:      {pf_time_total:.4f} segundos  ({pf_time_total*1000:.2f} ms)")
    print(f"PF Por Paso:   {pf_time_total/(n_steps-1)*1000:.4f} ms/paso")
    print(f"\nRelación PF/EKF: {pf_time_total/ekf_time_total:.2f}x")
    print("="*70)

    # ============================================================
# 6) Visualización - Formato Tesis
# ============================================================

# Configuración de formato para tesis
thesis_config = {
    'font_family': 'Computer Modern, serif',
    'font_size': 14,
    'title_font_size': 16,
    'legend_font_size': 13,
    'line_width_true': 2.5,
    'line_width_est': 2.0,
    'plot_width': 1000,
    'plot_height': 500,
    'grid_color': 'rgba(200, 200, 200, 0.3)',
    'grid_width': 0.5
}

# Cálculo de MSE por estado
mse_x_ekf = np.mean((x_true[:, 0] - x_hat_ekf[:, 0])**2)
mse_y_ekf = np.mean((x_true[:, 1] - x_hat_ekf[:, 1])**2)
mse_theta_ekf = np.mean((x_true[:, 2] - x_hat_ekf[:, 2])**2)
mse_x_pf = np.mean((x_true[:, 0] - x_hat_pf[:, 0])**2)
mse_y_pf = np.mean((x_true[:, 1] - x_hat_pf[:, 1])**2)
mse_theta_pf = np.mean((x_true[:, 2] - x_hat_pf[:, 2])**2)

mse_total_ekf = mse_x_ekf + mse_y_ekf + mse_theta_ekf
mse_total_pf = mse_x_pf + mse_y_pf + mse_theta_pf

# Reporte MSE
print("\n" + "="*70)
print("🏆 MEJOR FILTRO: ", end="")
mse_dict = {'EKF': mse_total_ekf, 'PF': mse_total_pf}
best_filter = min(mse_dict, key=mse_dict.get)
print(f"{best_filter} (MSE total: {mse_dict[best_filter]:.8f})")
for other_filter, mse_value in mse_dict.items():
    if other_filter != best_filter:
        print(f"{other_filter} MSE total: {mse_value:.8f}")
print("="*70)

print("\n--- Comparación de Desempeño (MSE) - Robot Diferencial ---")
print(f"EKF MSE x:      {mse_x_ekf:.8f}")
print(f"EKF MSE y:      {mse_y_ekf:.8f}")
print(f"EKF MSE θ:      {mse_theta_ekf:.8f}")
print(f"PF  MSE x:      {mse_x_pf:.8f}")
print(f"PF  MSE y:      {mse_y_pf:.8f}")
print(f"PF  MSE θ:      {mse_theta_pf:.8f}")



🔧 RHONN Configuration Set:
   ID: 2
   Name: Standard (Interactions)
   Number of features: 7

🎲 Semilla aleatoria (seed): 7517
   (Para reproducibilidad de resultados)


🔬 Particle Filter Configuration:
   Particles: 1500
   Optimization: ✅ ENABLED
      - Learning Rate: 1.5
      - Top-K Particles: 37

Starting Differential Robot simulation...
Simulación Completa.

⏱️  TIEMPOS DE ENTRENAMIENTO:
EKF Total:     0.0848 segundos  (84.81 ms)
EKF Por Paso:  0.0849 ms/paso

PF Total:      0.5803 segundos  (580.27 ms)
PF Por Paso:   0.5808 ms/paso

Relación PF/EKF: 6.84x

🏆 MEJOR FILTRO: EKF (MSE total: 0.00000005)
PF MSE total: 0.00000848

--- Comparación de Desempeño (MSE) - Robot Diferencial ---
EKF MSE x:      0.00000004
EKF MSE y:      0.00000000
EKF MSE θ:      0.00000000
PF  MSE x:      0.00000317
PF  MSE y:      0.00000264
PF  MSE θ:      0.00000267


In [14]:
# ============================================================
# OPTIMIZATION OF FILTER PARAMETERS - DIFFERENTIAL ROBOT
# ============================================================
# This section implements optimization algorithms to find the best
# hyperparameters for both EKF and PF filters for the differential robot

from scipy.optimize import differential_evolution, minimize
import warnings
warnings.filterwarnings('ignore')

def simulate_with_params(params, filter_type='EKF', n_steps=500, verbose=False):
    """
    Run simulation with given parameters and return MSE.
    
    Parameters for EKF: [Q_init, R_init, P_init, eta]
    Parameters for PF: [Q_std_x, Q_std_y, Q_std_theta, R_std_x, R_std_y, R_std_theta, n_particles]
    
    Returns:
    --------
    float : Total MSE (lower is better)
    """
    # Set reproducible seed
    np.random.seed(7517)
    
    # Simulation settings
    dt = 0.02
    process_noise_type = 'laplace'
    process_noise_std = 0.02
    measurement_noise_std = 0.05
    
    # True system initialization
    x_true = np.zeros((n_steps, 3))
    x_true[0] = [0.0, 0.0, 0.0]
    
    # Control inputs
    u_history = np.zeros((n_steps, 2))
    for k in range(n_steps):
        t = k * dt
        v = 0.5 + 0.3 * np.sin(0.5 * t)
        omega = 0.3 * np.cos(0.3 * t)
        u_history[k] = [v, omega]
    
    num_neurons = 3
    common_initial_weights = [np.random.uniform(-1.0, 1.0, num_weights_per_neuron) for _ in range(num_neurons)]
    
    try:
        if filter_type == 'EKF':
            Q_init, R_init, P_init, eta = params
            
            # Create EKF trainer with parameters
            trainer = EKF_RHONN_Trainer(
                num_neurons, num_weights_per_neuron,
                initial_weights=common_initial_weights,
                eta=eta, Q_init=Q_init, R_init=R_init, P_init=P_init
            )
            
            x_hat = np.zeros((n_steps, 3))
            x_hat[0] = x_true[0]
            
            # Run simulation
            for k in range(n_steps - 1):
                u_k = u_history[k]
                x_true[k+1] = plant(x_true[k], u_k, dt, process_noise_type, process_noise_std) + \
                              np.random.normal(0, measurement_noise_std, size=3)
                
                chi_k = x_true[k]
                chi_kp1 = x_true[k+1]
                
                trainer.update(chi_kp1, chi_k, x_hat[k])
                z = construct_z_vector(chi_k)
                for i in range(3):
                    x_hat[k+1, i] = np.dot(trainer.weights[i], z)
            
        else:  # PF
            Q_std_x, Q_std_y, Q_std_theta, R_std_x, R_std_y, R_std_theta, n_particles = params
            n_particles = int(n_particles)
            
            Q_std_per_neuron = [Q_std_x, Q_std_y, Q_std_theta]
            R_std_per_neuron = [R_std_x, R_std_y, R_std_theta]
            
            # Create PF trainer with parameters
            trainer = PF_RHONN_Trainer(
                num_neurons, num_weights_per_neuron,
                n_particles=n_particles,
                initial_weights=common_initial_weights,
                Q_std=Q_std_per_neuron,
                R_std=R_std_per_neuron,
                ess_threshold=0.5*n_particles,
                enable_optimization=False
            )
            
            # Initialize particles
            for i in range(num_neurons):
                trainer.particles[i] = np.tile(common_initial_weights[i], (trainer.n_particles, 1))
                trainer.particles[i] += np.random.normal(size=(trainer.n_particles, num_weights_per_neuron)) * 0.1
            
            x_hat = np.zeros((n_steps, 3))
            x_hat[0] = x_true[0]
            
            # Run simulation
            for k in range(n_steps - 1):
                u_k = u_history[k]
                x_true[k+1] = plant(x_true[k], u_k, dt, process_noise_type, process_noise_std) + \
                              np.random.normal(0, measurement_noise_std, size=3)
                
                chi_k = x_true[k]
                chi_kp1 = x_true[k+1]
                
                trainer.update(chi_kp1, chi_k, x_hat[k])
                est_w = trainer.get_estimate()
                z = construct_z_vector(chi_k)
                for i in range(3):
                    x_hat[k+1, i] = np.dot(est_w[i], z)
        
        # Calculate MSE
        mse_total = np.mean((x_true - x_hat)**2)
        
        if verbose:
            print(f"  Params: {params} → MSE: {mse_total:.6f}")
        
        return mse_total
        
    except Exception as e:
        if verbose:
            print(f"  Error with params {params}: {str(e)}")
        return 1e10  # Return large value on error


def optimize_ekf_parameters(n_iterations=50, n_steps=500):
    """
    Optimize EKF parameters using Differential Evolution.
    
    Parameters to optimize:
    - Q_init: Process noise covariance initialization
    - R_init: Measurement noise covariance initialization  
    - P_init: Initial state covariance
    - eta: Learning rate
    """
    print("\n" + "="*70)
    print("🔍 OPTIMIZING EKF PARAMETERS - DIFFERENTIAL ROBOT")
    print("="*70)
    print(f"Method: Differential Evolution")
    print(f"Iterations: {n_iterations}")
    print(f"Simulation steps: {n_steps}")
    print(f"Seed: 7517")
    print("="*70 + "\n")
    
    # Parameter bounds: [Q_init, R_init, P_init, eta]
    bounds = [
        (1e-5, 1.0),     # Q_init
        (1e-7, 1e-2),    # R_init
        (0.1, 10.0),     # P_init
        (0.1, 5.0)       # eta
    ]
    
    result = differential_evolution(
        lambda params: simulate_with_params(params, filter_type='EKF', n_steps=n_steps, verbose=False),
        bounds,
        seed=7517,
        maxiter=n_iterations,
        popsize=15,
        atol=1e-6,
        tol=1e-6,
        workers=1,
        updating='immediate',
        disp=True
    )
    
    print("\n" + "="*70)
    print("✅ EKF OPTIMIZATION COMPLETE")
    print("="*70)
    print(f"Best MSE: {result.fun:.8f}")
    print(f"\nOptimal Parameters:")
    print(f"  Q_init: {result.x[0]:.6e}")
    print(f"  R_init: {result.x[1]:.6e}")
    print(f"  P_init: {result.x[2]:.6f}")
    print(f"  eta:    {result.x[3]:.6f}")
    print("="*70 + "\n")
    
    return result


def optimize_pf_parameters(n_iterations=50, n_steps=500):
    """
    Optimize PF parameters using Differential Evolution.
    
    Parameters to optimize:
    - Q_std_x: Process noise std for x
    - Q_std_y: Process noise std for y
    - Q_std_theta: Process noise std for theta
    - R_std_x: Measurement noise std for x
    - R_std_y: Measurement noise std for y
    - R_std_theta: Measurement noise std for theta
    - n_particles: Number of particles
    """
    print("\n" + "="*70)
    print("🔍 OPTIMIZING PF PARAMETERS - DIFFERENTIAL ROBOT")
    print("="*70)
    print(f"Method: Differential Evolution")
    print(f"Iterations: {n_iterations}")
    print(f"Simulation steps: {n_steps}")
    print(f"Seed: 7517")
    print("="*70 + "\n")
    
    # Parameter bounds: [Q_std_x, Q_std_y, Q_std_theta, R_std_x, R_std_y, R_std_theta, n_particles]
    bounds = [
        (0.1, 5.0),      # Q_std_x
        (0.1, 5.0),      # Q_std_y
        (0.1, 5.0),      # Q_std_theta
        (0.001, 0.5),    # R_std_x
        (0.001, 0.5),    # R_std_y
        (0.001, 0.5),    # R_std_theta
        (300, 1500)      # n_particles
    ]
    
    result = differential_evolution(
        lambda params: simulate_with_params(params, filter_type='PF', n_steps=n_steps, verbose=False),
        bounds,
        seed=7517,
        maxiter=n_iterations,
        popsize=15,
        atol=1e-6,
        tol=1e-6,
        workers=1,
        updating='immediate',
        disp=True
    )
    
    print("\n" + "="*70)
    print("✅ PF OPTIMIZATION COMPLETE")
    print("="*70)
    print(f"Best MSE: {result.fun:.8f}")
    print(f"\nOptimal Parameters:")
    print(f"  Q_std_x:      {result.x[0]:.6f}")
    print(f"  Q_std_y:      {result.x[1]:.6f}")
    print(f"  Q_std_theta:  {result.x[2]:.6f}")
    print(f"  R_std_x:      {result.x[3]:.6f}")
    print(f"  R_std_y:      {result.x[4]:.6f}")
    print(f"  R_std_theta:  {result.x[5]:.6f}")
    print(f"  n_particles:  {int(result.x[6])}")
    print("="*70 + "\n")
    
    return result


def grid_search_ekf(n_steps=500):
    """
    Grid search for EKF parameters (faster but less thorough).
    """
    print("\n" + "="*70)
    print("🔍 GRID SEARCH FOR EKF PARAMETERS - DIFFERENTIAL ROBOT")
    print("="*70)
    print(f"Seed: 7517")
    print("="*70 + "\n")
    
    # Define parameter grid
    Q_init_values = [1e-4, 1e-3, 1e-2, 1e-1]
    R_init_values = [1e-7, 1e-6, 1e-5, 1e-4]
    P_init_values = [0.1, 0.5, 1.0, 5.0]
    eta_values = [0.5, 1.0, 2.0]
    
    best_mse = float('inf')
    best_params = None
    
    total_combinations = len(Q_init_values) * len(R_init_values) * len(P_init_values) * len(eta_values)
    current = 0
    
    for Q_init in Q_init_values:
        for R_init in R_init_values:
            for P_init in P_init_values:
                for eta in eta_values:
                    current += 1
                    params = [Q_init, R_init, P_init, eta]
                    mse = simulate_with_params(params, filter_type='EKF', n_steps=n_steps)
                    
                    if current % 10 == 0:
                        print(f"Progress: {current}/{total_combinations} ({current/total_combinations*100:.1f}%) - Best MSE: {best_mse:.6f}")
                    
                    if mse < best_mse:
                        best_mse = mse
                        best_params = params
                        print(f"  🌟 New best! MSE: {best_mse:.6f}, Params: {params}")
    
    print("\n" + "="*70)
    print("✅ GRID SEARCH COMPLETE")
    print("="*70)
    print(f"Best MSE: {best_mse:.8f}")
    print(f"\nOptimal Parameters:")
    print(f"  Q_init: {best_params[0]:.6e}")
    print(f"  R_init: {best_params[1]:.6e}")
    print(f"  P_init: {best_params[2]:.6f}")
    print(f"  eta:    {best_params[3]:.6f}")
    print("="*70 + "\n")
    
    return {'x': best_params, 'fun': best_mse}


# Example usage:
print("="*70)
print("🎯 FILTER PARAMETER OPTIMIZATION - DIFFERENTIAL ROBOT")
print("="*70)
print("Choose optimization method:")
print("  1. optimize_ekf_parameters()    - Optimize EKF (Differential Evolution)")
print("  2. optimize_pf_parameters()     - Optimize PF (Differential Evolution)")
print("  3. grid_search_ekf()            - Grid Search for EKF (faster)")
print("="*70)
print("\nTo run optimization, uncomment one of the following lines:")
print("# ekf_result = optimize_ekf_parameters(n_iterations=30, n_steps=500)")
print("# pf_result = optimize_pf_parameters(n_iterations=30, n_steps=500)")
print("# ekf_result = grid_search_ekf(n_steps=500)")
print("="*70)


🎯 FILTER PARAMETER OPTIMIZATION - DIFFERENTIAL ROBOT
Choose optimization method:
  1. optimize_ekf_parameters()    - Optimize EKF (Differential Evolution)
  2. optimize_pf_parameters()     - Optimize PF (Differential Evolution)
  3. grid_search_ekf()            - Grid Search for EKF (faster)

To run optimization, uncomment one of the following lines:
# ekf_result = optimize_ekf_parameters(n_iterations=30, n_steps=500)
# pf_result = optimize_pf_parameters(n_iterations=30, n_steps=500)
# ekf_result = grid_search_ekf(n_steps=500)


In [15]:
# ============================================================
# RUN OPTIMIZATION (Uncomment to execute)
# ============================================================
# This cell runs the parameter optimization for the differential robot.
# Uncomment the method you want to use:

# # Option 1: Optimize EKF parameters with Differential Evolution (recommended)
# ekf_opt_result = optimize_ekf_parameters(n_iterations=30, n_steps=500)

# # Option 2: Optimize PF parameters with Differential Evolution
# pf_opt_result = optimize_pf_parameters(n_iterations=30, n_steps=500)

# Option 3: Fast grid search for EKF
# ekf_opt_result = grid_search_ekf(n_steps=500)

# Option 4: Run both optimizations
# ekf_opt_result = optimize_ekf_parameters(n_iterations=30, n_steps=500)
# pf_opt_result = optimize_pf_parameters(n_iterations=30, n_steps=500)

print("✅ Optimization cell ready. Uncomment the desired method to run.")


✅ Optimization cell ready. Uncomment the desired method to run.


In [16]:
# ============================================================
# APPLY OPTIMIZED PARAMETERS TO SIMULATION - DIFFERENTIAL ROBOT
# ============================================================
# This function allows you to run a simulation with optimized parameters

def run_simulation_with_optimized_params(ekf_params=None, pf_params=None, n_steps=1000):
    """
    Run the full simulation with optimized parameters.
    
    Parameters:
    -----------
    ekf_params : list or None
        [Q_init, R_init, P_init, eta] for EKF
    pf_params : list or None
        [Q_std_x, Q_std_y, Q_std_theta, R_std_x, R_std_y, R_std_theta, n_particles] for PF
    n_steps : int
        Number of simulation steps
        
    Returns:
    --------
    dict : Results dictionary with timing, MSE, and state estimates
    """
    # Set seed for reproducibility
    np.random.seed(7517)
    print(f"🎲 Using seed: 7517\n")
    
    # Simulation settings
    dt = 0.02
    t_history = np.linspace(0, (n_steps-1) * dt, n_steps)
    
    process_noise_type = 'laplace' # 
    process_noise_std = 0.02
    measurement_noise_std = 0.05
    
    # True system initialization
    x_true = np.zeros((n_steps, 3))
    x_true[0] = [0.0, 0.0, 0.0]
    
    # Control inputs
    u_history = np.zeros((n_steps, 2))
    for k in range(n_steps):
        t = k * dt
        v = 0.5 + 0.3 * np.sin(0.5 * t)
        omega = 0.3 * np.cos(0.3 * t)
        u_history[k] = [v, omega]
    
    num_neurons = 3
    common_initial_weights = [np.random.uniform(-1.0, 1.0, num_weights_per_neuron) for _ in range(num_neurons)]
    
    # Initialize trainers with optimized or default parameters
    if ekf_params is not None:
        Q_init, R_init, P_init, eta = ekf_params
        print(f"📊 Using OPTIMIZED EKF parameters:")
        print(f"   Q_init: {Q_init:.6e}")
        print(f"   R_init: {R_init:.6e}")
        print(f"   P_init: {P_init:.6f}")
        print(f"   eta:    {eta:.6f}\n")
    else:
        Q_init, R_init, P_init, eta = 1e-3, 1e-7, 2.0, 1.0
        print(f"📊 Using DEFAULT EKF parameters\n")
    
    ekf_trainer = EKF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        initial_weights=common_initial_weights,
        eta=eta, Q_init=Q_init, R_init=R_init, P_init=P_init
    )
    
    if pf_params is not None:
        Q_std_x, Q_std_y, Q_std_theta, R_std_x, R_std_y, R_std_theta, n_particles = pf_params
        n_particles = int(n_particles)
        Q_std_per_neuron = [Q_std_x, Q_std_y, Q_std_theta]
        R_std_per_neuron = [R_std_x, R_std_y, R_std_theta]
        print(f"📊 Using OPTIMIZED PF parameters:")
        print(f"   Q_std:       [{Q_std_x:.4f}, {Q_std_y:.4f}, {Q_std_theta:.4f}]")
        print(f"   R_std:       [{R_std_x:.4f}, {R_std_y:.4f}, {R_std_theta:.4f}]")
        print(f"   n_particles: {n_particles}\n")
    else:
        n_particles = 1200
        Q_std_per_neuron = [1.0, 1.0, 1.0]
        R_std_per_neuron = [0.005, 0.005, 0.005]
        print(f"📊 Using DEFAULT PF parameters\n")
    
    pf_trainer = PF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        n_particles=n_particles,
        initial_weights=common_initial_weights,
        Q_std=Q_std_per_neuron,
        R_std=R_std_per_neuron,
        ess_threshold=0.5*n_particles,
        enable_optimization=True,
        opt_learning_rate=0.001*n_particles,
        opt_top_k=int(0.025*n_particles)
    )
    
    # Initialize PF particles
    for i in range(num_neurons):
        pf_trainer.particles[i] = np.tile(common_initial_weights[i], (pf_trainer.n_particles, 1))
        pf_trainer.particles[i] += np.random.normal(size=(pf_trainer.n_particles, num_weights_per_neuron)) * 0.1
    
    # Storage
    x_hat_ekf = np.zeros((n_steps, 3))
    x_hat_ekf[0] = x_true[0]
    x_hat_pf = np.zeros((n_steps, 3))
    x_hat_pf[0] = x_true[0]
    
    # Timing
    ekf_time_total = 0.0
    pf_time_total = 0.0
    
    print("🚀 Starting simulation...")
    for k in range(n_steps - 1):
        # Evolve true system
        u_k = u_history[k]
        x_true[k+1] = plant(x_true[k], u_k, dt, process_noise_type, process_noise_std) + \
                      np.random.normal(0, measurement_noise_std, size=3)
        
        chi_k = x_true[k]
        chi_kp1 = x_true[k+1]
        
        # EKF update
        t_start = time.perf_counter()
        ekf_trainer.update(chi_kp1, chi_k, x_hat_ekf[k])
        z_ekf = construct_z_vector(chi_k)
        for i in range(3):
            x_hat_ekf[k+1, i] = np.dot(ekf_trainer.weights[i], z_ekf)
        ekf_time_total += time.perf_counter() - t_start
        
        # PF update
        t_start = time.perf_counter()
        pf_trainer.update(chi_kp1, chi_k, x_hat_pf[k])
        est_w_pf = pf_trainer.get_estimate()
        z_pf = construct_z_vector(chi_k)
        for i in range(3):
            x_hat_pf[k+1, i] = np.dot(est_w_pf[i], z_pf)
        pf_time_total += time.perf_counter() - t_start
        
        if (k+1) % 200 == 0:
            print(f"  Step {k+1}/{n_steps-1}")
    
    print("✅ Simulation complete!\n")
    
    # Calculate MSE
    mse_ekf = np.mean((x_true - x_hat_ekf)**2)
    mse_pf = np.mean((x_true - x_hat_pf)**2)
    mse_x_ekf = np.mean((x_true[:, 0] - x_hat_ekf[:, 0])**2)
    mse_y_ekf = np.mean((x_true[:, 1] - x_hat_ekf[:, 1])**2)
    mse_theta_ekf = np.mean((x_true[:, 2] - x_hat_ekf[:, 2])**2)
    mse_x_pf = np.mean((x_true[:, 0] - x_hat_pf[:, 0])**2)
    mse_y_pf = np.mean((x_true[:, 1] - x_hat_pf[:, 1])**2)
    mse_theta_pf = np.mean((x_true[:, 2] - x_hat_pf[:, 2])**2)
    
    # Display results
    print("="*70)
    print("📊 RESULTS WITH OPTIMIZED PARAMETERS - DIFFERENTIAL ROBOT")
    print("="*70)
    print(f"\n⏱️  TIMING:")
    print(f"   EKF Total:    {ekf_time_total:.4f} s ({ekf_time_total/(n_steps-1)*1000:.4f} ms/step)")
    print(f"   PF Total:     {pf_time_total:.4f} s ({pf_time_total/(n_steps-1)*1000:.4f} ms/step)")
    print(f"   Ratio PF/EKF: {pf_time_total/ekf_time_total:.2f}x")
    
    print(f"\n🎯 MSE COMPARISON:")
    print(f"   EKF Total:    {mse_ekf:.8f}")
    print(f"     - x:        {mse_x_ekf:.8f}")
    print(f"     - y:        {mse_y_ekf:.8f}")
    print(f"     - θ:        {mse_theta_ekf:.8f}")
    print(f"   PF Total:     {mse_pf:.8f}")
    print(f"     - x:        {mse_x_pf:.8f}")
    print(f"     - y:        {mse_y_pf:.8f}")
    print(f"     - θ:        {mse_theta_pf:.8f}")
    
    winner = "EKF" if mse_ekf < mse_pf else "PF"
    improvement = abs(mse_ekf - mse_pf) / max(mse_ekf, mse_pf) * 100
    print(f"\n🏆 WINNER: {winner} (improvement: {improvement:.2f}%)")
    print("="*70 + "\n")
    
    return {
        't_history': t_history,
        'x_true': x_true,
        'x_hat_ekf': x_hat_ekf,
        'x_hat_pf': x_hat_pf,
        'mse_ekf': mse_ekf,
        'mse_pf': mse_pf,
        'timing_ekf': ekf_time_total,
        'timing_pf': pf_time_total
    }


# Example usage after optimization:
# results = run_simulation_with_optimized_params(
#     ekf_params=ekf_opt_result.x,  # From optimization
#     pf_params=pf_opt_result.x,    # From optimization
#     n_steps=1000
# )

print("✅ Function ready. Use run_simulation_with_optimized_params() to apply optimized parameters.")


✅ Function ready. Use run_simulation_with_optimized_params() to apply optimized parameters.


In [17]:


# Gráficas individuales por estado
states_info = [
    {'idx': 0, 'var': 'x', 'desc': 'Posición X', 'y_label': 'x (m)'},
    {'idx': 1, 'var': 'y', 'desc': 'Posición Y', 'y_label': 'y (m)'},
    {'idx': 2, 'var': 'θ', 'desc': 'Orientación', 'y_label': 'θ (rad)'}
]

for state_info in states_info:
    i = state_info['idx']
    
    fig = go.Figure()
    
    # Estado real (línea negra gruesa)
    fig.add_trace(go.Scatter(
        x=t_history, y=x_true[:, i],
        mode='lines',
        name='Estado Real',
        line=dict(color='#000000', width=thesis_config['line_width_true']),
        showlegend=True
    ))
    
    # Estimación EKF
    fig.add_trace(go.Scatter(
        x=t_history, y=x_hat_ekf[:, i],
        mode='lines',
        name='EKF-RHONN',
        line=dict(color='#1f77b4', width=thesis_config['line_width_est'], dash='dash'),
        showlegend=True
    ))
    
    # Estimación PF
    fig.add_trace(go.Scatter(
        x=t_history, y=x_hat_pf[:, i],
        mode='lines',
        name='PF-RHONN',
        line=dict(color='#d62728', width=thesis_config['line_width_est'], dash='dashdot'),
        showlegend=True
    ))
    
    fig.update_layout(
        title={
            'text': f'Estado {state_info["var"]}: {state_info["desc"]} - Robot Diferencial',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Tiempo (s)',
        yaxis_title=state_info['y_label'],
        xaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.98,
            y=0.98,
            xanchor='right',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.95)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=80, b=60)
    )
    
    fig.show()

# Trayectoria 2D (Robot en plano XY)
fig_traj = go.Figure()

# Trayectoria real
fig_traj.add_trace(go.Scatter(
    x=x_true[:, 0], y=x_true[:, 1],
    mode='lines+markers',
    name='Trayectoria Real',
    line=dict(color='#000000', width=3),
    marker=dict(size=4, color='black')
))

# Trayectoria EKF
fig_traj.add_trace(go.Scatter(
    x=x_hat_ekf[:, 0], y=x_hat_ekf[:, 1],
    mode='lines',
    name='Estimación EKF-RHONN',
    line=dict(color='#1f77b4', width=2, dash='dash')
))

# Trayectoria PF
fig_traj.add_trace(go.Scatter(
    x=x_hat_pf[:, 0], y=x_hat_pf[:, 1],
    mode='lines',
    name='Estimación PF-RHONN',
    line=dict(color='#d62728', width=2, dash='dashdot')
))

# Add start and end markers
fig_traj.add_trace(go.Scatter(
    x=[x_true[0, 0]], y=[x_true[0, 1]],
    mode='markers',
    name='Inicio',
    marker=dict(size=15, color='green', symbol='circle', line=dict(width=2, color='darkgreen'))
))

fig_traj.add_trace(go.Scatter(
    x=[x_true[-1, 0]], y=[x_true[-1, 1]],
    mode='markers',
    name='Final',
    marker=dict(size=15, color='red', symbol='square', line=dict(width=2, color='darkred'))
))

fig_traj.update_layout(
    title={
        'text': 'Trayectoria 2D - Robot Diferencial (Plano XY)',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='x (m)',
    yaxis_title='y (m)',
    xaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5,
        scaleanchor="x",
        scaleratio=1
    ),
    legend=dict(
        x=0.02,
        y=0.98,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.95)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_width'] * 0.75,
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_traj.show()

# Gráfica de barras comparando MSE
fig_mse = go.Figure()

filters = ['EKF-RHONN', 'PF-RHONN']

fig_mse.add_trace(go.Bar(
    name='Estado x',
    x=filters,
    y=[mse_x_ekf, mse_x_pf],
    marker_color='#636EFA',
    text=[f'{mse_x_ekf:.2e}', f'{mse_x_pf:.2e}'],
    textposition='outside'
))

fig_mse.add_trace(go.Bar(
    name='Estado y',
    x=filters,
    y=[mse_y_ekf, mse_y_pf],
    marker_color='#00CC96',
    text=[f'{mse_y_ekf:.2e}', f'{mse_y_pf:.2e}'],
    textposition='outside'
))

fig_mse.add_trace(go.Bar(
    name='Estado θ',
    x=filters,
    y=[mse_theta_ekf, mse_theta_pf],
    marker_color='#EF553B',
    text=[f'{mse_theta_ekf:.2e}', f'{mse_theta_pf:.2e}'],
    textposition='outside'
))

fig_mse.update_layout(
    title={
        'text': 'Comparación de Error Cuadrático Medio (MSE) - Robot Diferencial',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='Tipo de Filtro',
    yaxis_title='Error Cuadrático Medio (MSE)',
    yaxis=dict(
        type='log',
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    xaxis=dict(
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    legend=dict(
        x=0.98,
        y=0.98,
        xanchor='right',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.95)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
    ),
    barmode='group',
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_height'],
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_mse.show()
